# Tutorial 07 — Governance and Anomaly Detection

**No API key required. Fully deterministic.**

In a multi-tenant deployment, some tenants may behave abnormally — excessive rejection rates,
runaway cost utilisation, or spinning up too many concurrent jobs. eXo-brain provides two
independent governance layers to handle this:

1. **`detect_governance_anomalies`** — advisory-only detector; flags metrics that exceed
   configured thresholds without blocking any operation.
2. **`ByocFairAdmissionCoordinator`** — deterministic admission control; limits how many
   concurrent inflight requests are allowed globally, enforcing fairness across tenants.

Both are independent of the ingress gate chain from Tutorial 03.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

try:
    from dotenv import load_dotenv
    load_dotenv("../.env", override=False)
except ImportError:
    pass

## Part 1 — BYOC governance model

**BYOC** (Bring Your Own Compute) means customers use shared eXo-brain infrastructure
with their own configuration. Without governance:
- One tenant's runaway usage can starve others
- Silent rejection spikes go unnoticed
- Cost budgets are exceeded before anyone reacts

The two governance tools are complementary:
- Anomaly detector: **"something is wrong — take a look"**
- Admission coordinator: **"too many inflight — wait your turn"**

## Part 2 — Simulate 3 tenants

We define metric snapshots for three tenants:
- `tenant-a` — healthy usage
- `tenant-b` — healthy usage, slightly higher rejection rate
- `tenant-c` — anomalous: near-maximum cost utilisation and very high rejection rate

In [ ]:
from src.policies.governance_anomaly_detector import (
    detect_governance_anomalies,
    GovernanceAnomalyThresholds,
    GovernanceAnomaly,
)

# Shared thresholds for all tenants
thresholds = GovernanceAnomalyThresholds(
    cost_utilization_threshold=0.9,   # flag if > 90% of cost budget used
    rejection_rate_threshold=0.2,     # flag if > 20% of turns rejected
    reason_share_threshold=0.6,       # flag if one rejection reason > 60% of all rejections
    min_submit_attempts=5,
    min_rejection_count=3,
)

tenant_metrics = {
    "tenant-a": {
        "cost_utilization_ratio": 0.45,
        "rejection_rate": 0.05,
        "submit_attempts_total": 100,
        "rejected_results_total": 5,
        "rejection_reason_counts": {"POLICY_BLOCKED": 2, "TIMEOUT": 2, "RATE_LIMIT": 1},
    },
    "tenant-b": {
        "cost_utilization_ratio": 0.60,
        "rejection_rate": 0.18,
        "submit_attempts_total": 80,
        "rejected_results_total": 14,
        "rejection_reason_counts": {"POLICY_BLOCKED": 6, "TIMEOUT": 5, "RATE_LIMIT": 3},
    },
    "tenant-c": {
        "cost_utilization_ratio": 0.95,  # above threshold
        "rejection_rate": 0.90,          # well above threshold
        "submit_attempts_total": 200,
        "rejected_results_total": 180,
        "rejection_reason_counts": {"POLICY_BLOCKED": 160, "TIMEOUT": 20},
    },
}

print("Tenant metrics loaded for:", list(tenant_metrics.keys()))

## Part 3 — Run anomaly detection

`detect_governance_anomalies` is a pure function — no side effects, no blocking.
It returns a list of `GovernanceAnomaly` findings (empty list = healthy).

In [ ]:
for tenant_id, metrics in tenant_metrics.items():
    anomalies = detect_governance_anomalies(
        cost_utilization_ratio=metrics["cost_utilization_ratio"],
        rejection_rate=metrics["rejection_rate"],
        submit_attempts_total=metrics["submit_attempts_total"],
        rejected_results_total=metrics["rejected_results_total"],
        rejection_reason_counts=metrics["rejection_reason_counts"],
        thresholds=thresholds,
    )
    print(f"\n{tenant_id}: {len(anomalies)} anomaly/ies")
    for a in anomalies:
        print(f"  code      : {a.code}")
        print(f"  severity  : {a.severity}")
        print(f"  message   : {a.message}")
        print(f"  value     : {a.value:.2f}  threshold: {a.threshold:.2f}")

# Verify expected results
assert detect_governance_anomalies(
    cost_utilization_ratio=tenant_metrics["tenant-a"]["cost_utilization_ratio"],
    rejection_rate=tenant_metrics["tenant-a"]["rejection_rate"],
    submit_attempts_total=tenant_metrics["tenant-a"]["submit_attempts_total"],
    rejected_results_total=tenant_metrics["tenant-a"]["rejected_results_total"],
    rejection_reason_counts=tenant_metrics["tenant-a"]["rejection_reason_counts"],
    thresholds=thresholds,
) == [], "tenant-a should be clean"

c_anomalies = detect_governance_anomalies(
    cost_utilization_ratio=tenant_metrics["tenant-c"]["cost_utilization_ratio"],
    rejection_rate=tenant_metrics["tenant-c"]["rejection_rate"],
    submit_attempts_total=tenant_metrics["tenant-c"]["submit_attempts_total"],
    rejected_results_total=tenant_metrics["tenant-c"]["rejected_results_total"],
    rejection_reason_counts=tenant_metrics["tenant-c"]["rejection_reason_counts"],
    thresholds=thresholds,
)
assert len(c_anomalies) >= 2, f"tenant-c should have at least 2 anomalies, got {len(c_anomalies)}"
print("\nPASS — anomaly detection: tenant-a clean, tenant-c flagged")

## Part 4 — Fair admission: global inflight cap

`ByocFairAdmissionCoordinator(max_inflight_global=3)` allows at most 3 concurrent
inflight requests across all tenants. The 4th request returns `None` after timing out.

In [ ]:
import threading
import time
from src.policies.byoc_fairness import ByocFairAdmissionCoordinator, FairAdmissionToken

coordinator = ByocFairAdmissionCoordinator(max_inflight_global=3)

# Acquire 3 slots — all should succeed
token_a = coordinator.acquire(tenant_id="tenant-a", wait_timeout_ms=100)
token_b = coordinator.acquire(tenant_id="tenant-b", wait_timeout_ms=100)
token_c = coordinator.acquire(tenant_id="tenant-c", wait_timeout_ms=100)

print("token_a:", token_a)
print("token_b:", token_b)
print("token_c:", token_c)

assert token_a is not None, "slot 1 should be granted"
assert token_b is not None, "slot 2 should be granted"
assert token_c is not None, "slot 3 should be granted"
assert isinstance(token_a, FairAdmissionToken)

# 4th acquire — no slots available, times out → returns None
token_d = coordinator.acquire(tenant_id="tenant-a", wait_timeout_ms=50)
print("token_d (should be None):", token_d)
assert token_d is None, "4th acquire should time out when all 3 slots taken"

print("\nPASS — 3 slots granted, 4th timed out correctly")

## Part 5 — Inspect admission stats

In [ ]:
stats = coordinator.stats()
print("Admission stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

assert stats["fair_admission_inflight_total"] == 3
assert stats["fair_admission_max_inflight_global"] == 3
print("\nPASS — stats reflect 3 inflight slots taken")

## Part 6 — Release unblocks the next waiter

Releasing a token frees the slot. A waiting `acquire()` on another thread
will be granted the slot.

In [ ]:
# Release one token — slot becomes available
coordinator.release(token_a)
print("Released token_a")

stats_after = coordinator.stats()
print("Stats after release:")
for k, v in stats_after.items():
    print(f"  {k}: {v}")

# Now acquire should succeed again
token_e = coordinator.acquire(tenant_id="tenant-b", wait_timeout_ms=100)
print("token_e after release:", token_e)
assert token_e is not None, "slot should be available after release"

# Clean up remaining tokens
coordinator.release(token_b)
coordinator.release(token_c)
coordinator.release(token_e)

print("\nPASS — release + re-acquire works correctly")

## Part 7 — Per-tenant policy overlays

`TenantPolicyOverlayStore` maps tenant IDs to their policy configuration overlays.
This is the mechanism by which different tenants can have different ingress profiles,
classifier settings, and custom rules — without any shared mutable state.

In [ ]:
from src.tenancy.policy_overlay import TenantPolicyOverlayStore

overlay_store = TenantPolicyOverlayStore()

# Each tenant brings their own policy configuration
overlay_store.set_overlay("tenant-a", {
    "ingress_profile": "baseline",
    "ingress_classifier_mode": "off",
})

overlay_store.set_overlay("tenant-b", {
    "ingress_profile": "strict",
    "ingress_classifier_mode": "shadow",
    "ingress_classifier_threshold": 0.65,
})

overlay_store.set_overlay("tenant-c", {
    "ingress_profile": "hardened",
    "ingress_classifier_mode": "enforce",
    "ingress_classifier_threshold": 0.5,
    "ingress_custom_rules": [
        {
            "rule_id":    "c-block-001",
            "action":     "deny",
            "match_type": "contains_any",
            "patterns":   ["export all", "bypass limit"],
            "reason_code": "TENANT_C_BLOCKED",
            "message":    "This action is not permitted for your account.",
        }
    ],
})

for tid in ["tenant-a", "tenant-b", "tenant-c"]:
    overlay = overlay_store.get_overlay(tid)
    print(f"{tid}: profile={overlay.get('ingress_profile')}, "
          f"classifier={overlay.get('ingress_classifier_mode')}")

print("\nPASS — per-tenant overlays stored and retrieved independently")

## Summary

| Capability | Module | Key API |
|---|---|---|
| Anomaly detection | `src/policies/governance_anomaly_detector` | `detect_governance_anomalies(...)` |
| Anomaly thresholds | `src/policies/governance_anomaly_detector` | `GovernanceAnomalyThresholds` |
| Anomaly finding | `src/policies/governance_anomaly_detector` | `GovernanceAnomaly.code/severity/value/threshold` |
| Fair admission | `src/policies/byoc_fairness` | `ByocFairAdmissionCoordinator.acquire()` |
| Admission token | `src/policies/byoc_fairness` | `FairAdmissionToken` |
| Admission stats | `src/policies/byoc_fairness` | `coordinator.stats()` |
| Per-tenant config | `src/tenancy/policy_overlay` | `TenantPolicyOverlayStore.set_overlay()` |

**Key insight:** Anomaly detection is advisory — it never blocks. Fair admission is
deterministic — it blocks when the global limit is hit. Both are independent of the
ingress gate chain. Together they give operators visibility and control over
multi-tenant resource sharing.